In [ ]:
# ==========================================
# 实验：MiniRocket + LightGBM（内存优化版）
# ==========================================

!pip install lightgbm -q

import numpy as np
import pandas as pd
import pickle
import os
import time
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score
from sklearn.preprocessing import MinMaxScaler
from sktime.transformations.panel.rocket import MiniRocketMultivariate
import lightgbm as lgb
import warnings
warnings.filterwarnings('ignore')

print("="*60)
print("NGAFID 实验 - MiniRocket + LightGBM (内存优化版)")
print("="*60)

# ==========================================
# 1. 加载数据
# ==========================================
data_dir = '/root'

with open(os.path.join(data_dir, 'flight_data.pkl'), 'rb') as f:
    data = pickle.load(f)

header_df = pd.read_csv(os.path.join(data_dir, 'flight_header.csv'))

print(f"\n✅ 数据加载成功")

# ==========================================
# 2. 筛选完整19类基准子集
# ==========================================
mask = (abs(header_df['date_diff']) <= 2) & (header_df['date_diff'] != 0)
mask = mask & (header_df['label'].notna())

filtered_header = header_df[mask].copy()
filtered_header = filtered_header.reset_index(drop=True)
flight_ids = filtered_header['Master Index'].values

print(f"\n📊 筛选后航班数: {len(filtered_header)}")
print(f"  维护后 (0): {sum(filtered_header['before_after']==0)}")
print(f"  维护前 (1): {sum(filtered_header['before_after']==1)}")

# ==========================================
# 3. 准备特征和标签
# ==========================================
target_len = 4096
X_list = []
y = []

print("\n⏳ 准备数据...")

for idx, flight_id in enumerate(flight_ids):
    sensor_data = data[flight_id]
    sensor_data = np.nan_to_num(sensor_data, nan=0.0)
    
    if sensor_data.shape[0] >= target_len:
        sensor_data = sensor_data[-target_len:, :]
    else:
        pad_width = ((0, target_len - sensor_data.shape[0]), (0, 0))
        sensor_data = np.pad(sensor_data, pad_width, mode='constant', constant_values=0)
    
    X_list.append(sensor_data)
    label = filtered_header.iloc[idx]['before_after']
    y.append(label)

X = np.array(X_list, dtype=np.float32)
y = np.array(y)

print(f"\n✅ 数据准备完成")
print(f"  X 形状: {X.shape}")
print(f"  标签分布: 0={sum(y==0)}, 1={sum(y==1)}")

# ==========================================
# 4. 5折交叉验证（内存优化版）
# ==========================================
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# 🔥 减少核数量
minirocket = MiniRocketMultivariate(
    random_state=42,
    num_kernels=5000,  # 从 10000 减少到 5000
)

# 🔥 限制 LightGBM 内存使用
classifier = lgb.LGBMClassifier(
    n_estimators=50,       # 减少树的数量
    num_leaves=15,         # 减少叶子节点
    max_depth=5,           # 减少深度
    random_state=42,
    verbose=-1
)

accuracies, f1_scores, auc_scores = [], [], []

print("\n" + "="*60)
print("5折交叉验证（MiniRocket + LightGBM 内存优化版）")
print("="*60)

fold = 1
for train_idx, val_idx in skf.split(X, y):
    X_train, X_val = X[train_idx], X[val_idx]
    y_train, y_val = y[train_idx], y[val_idx]
    
    print(f"\n--- Fold {fold} ---")
    print(f"  训练集: {len(train_idx)} 样本 (0={sum(y_train==0)}, 1={sum(y_train==1)})")
    print(f"  验证集: {len(val_idx)} 样本 (0={sum(y_val==0)}, 1={sum(y_val==1)})")
    
    start = time.time()
    
    # fold内归一化
    n_samples, length, channels = X_train.shape
    X_train_flat = X_train.reshape(-1, channels)
    scaler = MinMaxScaler()
    scaler.fit(X_train_flat)
    
    X_train_norm = scaler.transform(X_train_flat).reshape(n_samples, length, channels)
    X_val_flat = X_val.reshape(-1, channels)
    X_val_norm = scaler.transform(X_val_flat).reshape(X_val.shape)
    
    # MiniRocket 特征提取
    X_train_transform = minirocket.fit_transform(X_train_norm, y_train)
    X_val_transform = minirocket.transform(X_val_norm)
    
    # LightGBM 训练
    classifier.fit(X_train_transform, y_train)
    
    y_pred = classifier.predict(X_val_transform)
    y_prob = classifier.predict_proba(X_val_transform)[:, 1]
    
    elapsed = time.time() - start
    
    acc = accuracy_score(y_val, y_pred)
    f1 = f1_score(y_val, y_pred)
    auc = roc_auc_score(y_val, y_prob)
    
    accuracies.append(acc)
    f1_scores.append(f1)
    auc_scores.append(auc)
    
    print(f"  准确率: {acc:.4f}, F1: {f1:.4f}, AUC: {auc:.4f}, 耗时: {elapsed:.2f}s")
    fold += 1

print("\n" + "="*60)
print("📊 MiniRocket + LightGBM 最终结果:")
print("="*60)
print(f"  准确率: {np.mean(accuracies):.4f} ± {np.std(accuracies):.4f}")
print(f"  F1分数: {np.mean(f1_scores):.4f} ± {np.std(f1_scores):.4f}")
print(f"  AUC:    {np.mean(auc_scores):.4f} ± {np.std(auc_scores):.4f}")
print("="*60)

print("\n📊 实验对比:")
print("-" * 60)
print(f"  逻辑回归 + 全局归一化:    ~54.3%")
print(f"  逻辑回归 + fold内归一化:  58.7%")
print(f"  LightGBM + fold内归一化:  {np.mean(accuracies):.4f}")
print("-" * 60)

NGAFID 实验 - MiniRocket + LightGBM (内存优化版)

✅ 数据加载成功

📊 筛选后航班数: 11446
  维护后 (0): 5844
  维护前 (1): 5602

⏳ 准备数据...


In [1]:
import os

# 检查关键文件是否存在
print("one_parq 存在:", os.path.exists('/root/all_flights/one_parq'))
print("flight_header.csv 存在:", os.path.exists('/root/all_flights/flight_header.csv'))

one_parq 存在: True
flight_header.csv 存在: True
